# Attention Evaluation - Nepali

In [1]:
import argparse
import os
import math
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import sentencepiece as spm
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ==========================================
# DEFAULT PATHS (Fallback / Kaggle)
# ==========================================
LANG = "Nepali"
PRETRAINED_CHECKPOINT = "/kaggle/input/notebooks/srinivasu2005/train-nepali/checkpoints_nepali/best_model.pt"
FINETUNED_CHECKPOINT = "/kaggle/input/notebooks/srinivasu2005/finetune-nepali-path/finetuned_model_nepali_7k/best_finetuned_model.pt"
TEST_JSONL = "/kaggle/input/datasets/srinivasu2005/finetune-nepali/test.jsonl"
TOKENIZER_PATH = "/kaggle/input/notebooks/srinivasu2005/nepali-tokenizer/tokenizer/tokenizer.model"
OUTPUT_DIR = "eval_nepali_finetuned/attention"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEFAULT_FONT_PATH = "/tmp/NotoSansDevanagari-Regular.ttf"

def setup_devanagari_font(font_path=None):
    """Safely loads and registers Devanagari font with fallback to DejaVu Sans."""
    import logging
    logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

    candidate_paths = [
        font_path,
        DEFAULT_FONT_PATH,
        os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "fonts", "Mukta-Regular.ttf") if "__file__" in globals() else "",
        os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "fonts", "NotoSansDevanagari-Regular.ttf") if "__file__" in globals() else "",
        "/usr/share/fonts/truetype/noto/NotoSansDevanagari-Regular.ttf",
        "/tmp/Mukta-Regular.ttf",
    ]
    loaded_font_name = None
    for path in candidate_paths:
        if path and os.path.isfile(path) and os.path.getsize(path) > 0:
            try:
                fm.fontManager.addfont(path)
                loaded_font_name = fm.FontProperties(fname=path).get_name()
                break
            except Exception:
                pass

    if not loaded_font_name:
        try:
            import urllib.request
            url = "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansDevanagari/NotoSansDevanagari-Regular.ttf"
            urllib.request.urlretrieve(url, DEFAULT_FONT_PATH)
            if os.path.isfile(DEFAULT_FONT_PATH) and os.path.getsize(DEFAULT_FONT_PATH) > 0:
                fm.fontManager.addfont(DEFAULT_FONT_PATH)
                loaded_font_name = fm.FontProperties(fname=DEFAULT_FONT_PATH).get_name()
        except Exception as e:
            pass

    font_families = []
    if loaded_font_name:
        font_families.append(loaded_font_name)
    font_families.extend(['Noto Sans Devanagari', 'DejaVu Sans'])

    plt.rcParams['font.family'] = font_families
    plt.rcParams['axes.unicode_minus'] = False

setup_devanagari_font()

## Model Architecture

In [2]:
VOCAB_SIZE = 4000
D_FF = 1184

@dataclass
class TransformerConfig:
    vocab_size: int = VOCAB_SIZE; d_model: int = 512; n_layers: int = 8
    n_heads: int = 8; d_ff: int = D_FF; max_seq_len: int = 512
    dropout: float = 0.0; tie_weights: bool = True; rope_base: int = 10000
    pad_id: int = 0; bos_id: int = 2; eos_id: int = 3
    @property
    def d_k(self): return self.d_model // self.n_heads

def precompute_rope_frequencies(d_k, max_seq_len, base=10000, device=None):
    freq = torch.arange(0, d_k, 2, dtype=torch.float32, device=device)
    theta = 1.0 / (base ** (freq / d_k))
    pos = torch.arange(0, max_seq_len, dtype=torch.float32, device=device)
    angles = torch.outer(pos, theta)
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    c = cos.unsqueeze(0).unsqueeze(0); s = sin.unsqueeze(0).unsqueeze(0)
    return torch.stack([x1*c - x2*s, x1*s + x2*c], dim=-1).flatten(-2)

class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads, self.d_k, self.d_model = cfg.n_heads, cfg.d_k, cfg.d_model
        self.W_Q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_K = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_V = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_O = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        cos, sin = precompute_rope_frequencies(cfg.d_k, cfg.max_seq_len, cfg.rope_base)
        self.register_buffer("cos_table", cos); self.register_buffer("sin_table", sin)
        self.register_buffer("causal_mask", torch.triu(torch.ones(cfg.max_seq_len, cfg.max_seq_len), diagonal=1).bool())
    def forward(self, x, return_attention=False):
        B, T, _ = x.shape
        Q = self.W_Q(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        K = self.W_K(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        V = self.W_V(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        c, s = self.cos_table[:T], self.sin_table[:T]
        Q, K = apply_rope(Q,c,s), apply_rope(K,c,s)
        sc = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        sc = sc.masked_fill(self.causal_mask[:T,:T].unsqueeze(0).unsqueeze(0), float('-inf'))
        aw = torch.softmax(sc, dim=-1)
        ctx = torch.matmul(self.attn_dropout(aw), V).transpose(1,2).contiguous().view(B,T,self.d_model)
        return (self.resid_dropout(self.W_O(ctx)), aw) if return_attention else (self.resid_dropout(self.W_O(ctx)), None)

class SwiGLUFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.W_gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)
    def forward(self, x): return self.dropout(self.W_down(F.silu(self.W_gate(x)) * self.W_up(x)))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model); self.attn = MultiHeadCausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model); self.ffn = SwiGLUFeedForward(cfg)
    def forward(self, x, return_attention=False):
        ao, aw = self.attn(self.ln1(x), return_attention); x = x + ao; x = x + self.ffn(self.ln2(x)); return x, aw

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.emb_dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_ln = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_weights: self.lm_head.weight = self.embedding.weight
    def forward(self, input_ids, targets=None, return_attention=False):
        x = self.emb_dropout(self.embedding(input_ids))
        all_attn = [] if return_attention else None
        for block in self.blocks:
            x, aw = block(x, return_attention)
            if return_attention and aw is not None: all_attn.append(aw)
        logits = self.lm_head(self.final_ln(x))
        result = {"logits": logits}
        if targets is not None:
            result["loss"] = F.cross_entropy(logits.view(-1, self.config.vocab_size), targets.view(-1), ignore_index=self.config.pad_id)
        if return_attention: result["attention_weights"] = all_attn
        return result
    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=128, temperature=0.0, top_k=0, repetition_penalty=1.0):
        self.eval(); gen = input_ids
        for _ in range(max_new_tokens):
            ctx = gen if gen.shape[1] <= self.config.max_seq_len else gen[:, -self.config.max_seq_len:]
            logits = self.forward(ctx)["logits"][:, -1, :]
            if repetition_penalty != 1.0:
                for b in range(input_ids.size(0)):
                    for token_id in set(gen[b].tolist()):
                        if logits[b, token_id] < 0: logits[b, token_id] *= repetition_penalty
                        else: logits[b, token_id] /= repetition_penalty
            if temperature == 0.0: nt = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = float('-inf')
                nt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            gen = torch.cat([gen, nt], dim=1)
            if (nt == self.config.eos_id).all(): break
        return gen

## Attention Analysis Logic

In [3]:
def load_model(checkpoint_path, device="cpu"):
    cfg = TransformerConfig(dropout=0.0)
    model = DecoderOnlyTransformer(cfg).to(device)
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        except TypeError:
            checkpoint = torch.load(checkpoint_path, map_location=device)
        state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
        model.load_state_dict(state_dict)
    else:
        print(f"Warning: Checkpoint not found at '{checkpoint_path}'. Evaluating uninitialized model.")
    model.eval()
    return model

def compute_entropy(attn_matrix):
    a = attn_matrix.clamp(min=1e-10)
    return -(a * a.log()).sum(dim=-1).mean().item()

def compute_mean_distance(attn_matrix):
    seq_len = attn_matrix.shape[-1]
    positions = torch.arange(seq_len, device=attn_matrix.device, dtype=torch.float32)
    dist_matrix = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()
    return (attn_matrix * dist_matrix).sum(dim=-1).mean().item()

def run_attention_analysis(args=None):
    pretrained_checkpoint = getattr(args, "pretrained_checkpoint", None) or PRETRAINED_CHECKPOINT
    finetuned_checkpoint = getattr(args, "finetuned_checkpoint", None) or FINETUNED_CHECKPOINT
    test_jsonl = getattr(args, "test_data", None) or getattr(args, "test_jsonl", None) or TEST_JSONL
    tokenizer_path = getattr(args, "tokenizer_path", None) or getattr(args, "tokenizer", None) or TOKENIZER_PATH
    output_dir = getattr(args, "output_dir", None) or OUTPUT_DIR
    num_examples = getattr(args, "num_examples", None) if getattr(args, "num_examples", None) is not None else 3
    font_path = getattr(args, "font_path", None)
    device_name = getattr(args, "device", None)

    if device_name:
        device = torch.device(device_name)
    else:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    os.makedirs(output_dir, exist_ok=True)
    setup_devanagari_font(font_path)

    if not os.path.exists(tokenizer_path):
        print(f"Error: Tokenizer not found at: {tokenizer_path}")
        return
    sp = spm.SentencePieceProcessor()
    sp.load(tokenizer_path)
    print(f"Loaded tokenizer from: {tokenizer_path}")

    models = {
        "pretrained": load_model(pretrained_checkpoint, device),
        "finetuned": load_model(finetuned_checkpoint, device)
    }

    examples = []
    if os.path.exists(test_jsonl):
        with open(test_jsonl, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    examples.append(json.loads(line))
                    if num_examples and len(examples) >= num_examples:
                        break
        print(f"Loaded {len(examples)} examples from {test_jsonl} for attention analysis.")
    else:
        print(f"Warning: {test_jsonl} not found. Running with empty list.")

    all_data = {}

    for ex_idx, ex in enumerate(examples):
        inp = ex.get("input", ex.get("question", "")).strip()
        tgt = ex.get("target", ex.get("answer", "")).strip()
        if "उत्तर:" in tgt:
            full_text = f"{inp} {tgt}"
        elif not inp.endswith("उत्तर:"):
            full_text = f"{inp} उत्तर: {tgt}"
        else:
            full_text = f"{inp} {tgt}"

        bos = sp.bos_id() if sp.bos_id() != -1 else 2
        input_ids = [bos] + sp.encode(full_text)
        tokens = [sp.id_to_piece(t) for t in input_ids]
        input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)

        ex_data = {"pretrained": {}, "finetuned": {}, "tokens": tokens}

        for model_name, model in models.items():
            with torch.no_grad():
                outputs = model(input_tensor, return_attention=True)

            attns = [aw[0].cpu() for aw in outputs["attention_weights"]]
            n_layers = len(attns)
            n_heads = attns[0].shape[0] if n_layers > 0 else 8
            entropies = np.zeros((n_layers, n_heads))
            distances = np.zeros((n_layers, n_heads))

            for l_idx, layer_attn in enumerate(attns):
                for h_idx in range(n_heads):
                    h_attn = layer_attn[h_idx]
                    entropies[l_idx, h_idx] = compute_entropy(h_attn)
                    distances[l_idx, h_idx] = compute_mean_distance(h_attn)

            # 1. Attention Heatmaps for selected layers and heads (early, middle, late)
            layers_to_plot = [0, n_layers // 2, n_layers - 1]
            for l_idx in layers_to_plot:
                if l_idx < n_layers:
                    for h_idx in range(min(4, n_heads)):
                        a = attns[l_idx][h_idx].numpy()
                        fig, ax = plt.subplots(figsize=(10, 8))
                        im = ax.imshow(a, cmap="Blues", aspect="auto", vmin=0)
                        if len(tokens) <= 40:
                            ax.set_xticks(range(len(tokens)))
                            ax.set_yticks(range(len(tokens)))
                            ax.set_xticklabels(tokens, rotation=90, fontsize=6)
                            ax.set_yticklabels(tokens, fontsize=6)
                        ax.set_xlabel("Key")
                        ax.set_ylabel("Query")
                        ax.set_title(f"{LANG} ({model_name.capitalize()}) - Ex {ex_idx} - Layer {l_idx}, Head {h_idx}")
                        plt.colorbar(im, ax=ax)
                        plt.tight_layout()
                        plt.savefig(os.path.join(output_dir, f"attn_{model_name}_ex{ex_idx}_L{l_idx}_H{h_idx}.png"), dpi=150)
                        plt.close()

            ex_data[model_name]["entropy"] = entropies.tolist()
            ex_data[model_name]["distance"] = distances.tolist()

            # 2. Entropy and Mean Distance Summary Heatmaps for this model
            for mat, name, cmap, label in [
                (entropies, f"entropy_{model_name}_ex{ex_idx}", "YlOrRd", "Entropy (nats)"),
                (distances, f"distance_{model_name}_ex{ex_idx}", "viridis", "Mean Distance (tokens)")
            ]:
                fig, ax = plt.subplots(figsize=(8, 6))
                im = ax.imshow(mat, cmap=cmap, aspect="auto")
                ax.set_xlabel("Head")
                ax.set_ylabel("Layer")
                ax.set_title(f"{LANG} ({model_name.capitalize()}) {label}")
                ax.set_xticks(range(n_heads))
                ax.set_yticks(range(n_layers))
                plt.colorbar(im, ax=ax, label=label)
                plt.tight_layout()
                plt.savefig(os.path.join(output_dir, f"{name}.png"), dpi=150)
                plt.close()

        # 3. Delta (Finetuned - Pretrained) Summary Heatmaps
        delta_ent = np.array(ex_data["finetuned"]["entropy"]) - np.array(ex_data["pretrained"]["entropy"])
        delta_dist = np.array(ex_data["finetuned"]["distance"]) - np.array(ex_data["pretrained"]["distance"])
        ex_data["delta_entropy"] = delta_ent.tolist()
        ex_data["delta_distance"] = delta_dist.tolist()

        for mat, name, cmap, label in [
            (delta_ent, f"delta_entropy_ex{ex_idx}", "RdBu_r", "Delta Entropy (nats)"),
            (delta_dist, f"delta_distance_ex{ex_idx}", "RdBu_r", "Delta Mean Distance (tokens)")
        ]:
            fig, ax = plt.subplots(figsize=(8, 6))
            max_val = max(abs(float(mat.min())), abs(float(mat.max()))) if mat.size > 0 else 1.0
            max_val = max_val if max_val > 0 else 1.0
            im = ax.imshow(mat, cmap=cmap, aspect="auto", vmin=-max_val, vmax=max_val)
            ax.set_xlabel("Head")
            ax.set_ylabel("Layer")
            ax.set_title(f"{LANG} {label} (Finetuned - Pretrained) - Ex {ex_idx}")
            ax.set_xticks(range(n_heads))
            ax.set_yticks(range(n_layers))
            plt.colorbar(im, ax=ax, label=label)
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"{name}.png"), dpi=150)
            plt.close()

        all_data[f"ex_{ex_idx}"] = ex_data

    # Overall summary across multiple examples if applicable
    if len(examples) > 1:
        for model_name in ["pretrained", "finetuned"]:
            avg_ent = np.mean([all_data[f"ex_{i}"][model_name]["entropy"] for i in range(len(examples))], axis=0)
            avg_dist = np.mean([all_data[f"ex_{i}"][model_name]["distance"] for i in range(len(examples))], axis=0)

            for mat, name, cmap, label in [
                (avg_ent, f"entropy_{model_name}_summary", "YlOrRd", "Entropy (nats)"),
                (avg_dist, f"distance_{model_name}_summary", "viridis", "Mean Distance (tokens)")
            ]:
                fig, ax = plt.subplots(figsize=(8, 6))
                im = ax.imshow(mat, cmap=cmap, aspect="auto")
                ax.set_xlabel("Head")
                ax.set_ylabel("Layer")
                ax.set_title(f"{LANG} ({model_name.capitalize()}) {label} (Average)")
                ax.set_xticks(range(n_heads))
                ax.set_yticks(range(n_layers))
                plt.colorbar(im, ax=ax, label=label)
                plt.tight_layout()
                plt.savefig(os.path.join(output_dir, f"{name}.png"), dpi=150)
                plt.close()

        avg_delta_ent = np.mean([all_data[f"ex_{i}"]["delta_entropy"] for i in range(len(examples))], axis=0)
        avg_delta_dist = np.mean([all_data[f"ex_{i}"]["delta_distance"] for i in range(len(examples))], axis=0)

        for mat, name, cmap, label in [
            (avg_delta_ent, "delta_entropy_summary", "RdBu_r", "Delta Entropy (nats)"),
            (avg_delta_dist, "delta_distance_summary", "RdBu_r", "Delta Mean Distance (tokens)")
        ]:
            fig, ax = plt.subplots(figsize=(8, 6))
            max_val = max(abs(float(mat.min())), abs(float(mat.max()))) if mat.size > 0 else 1.0
            max_val = max_val if max_val > 0 else 1.0
            im = ax.imshow(mat, cmap=cmap, aspect="auto", vmin=-max_val, vmax=max_val)
            ax.set_xlabel("Head")
            ax.set_ylabel("Layer")
            ax.set_title(f"{LANG} {label} (Finetuned - Pretrained) (Average)")
            ax.set_xticks(range(n_heads))
            ax.set_yticks(range(n_layers))
            plt.colorbar(im, ax=ax, label=label)
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"{name}.png"), dpi=150)
            plt.close()

    metrics_path = os.path.join(output_dir, "attention_metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)
    print(f"Attention analysis complete. Metrics and heatmaps saved to: {output_dir}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Attention Analysis: Pretrained vs Finetuned Nepali Model")
    parser.add_argument("--pretrained_checkpoint", type=str, default=PRETRAINED_CHECKPOINT, help="Path to pretrained model checkpoint")
    parser.add_argument("--finetuned_checkpoint", type=str, default=FINETUNED_CHECKPOINT, help="Path to finetuned model checkpoint")
    parser.add_argument("--test_data", "--test_jsonl", dest="test_data", type=str, default=TEST_JSONL, help="Path to test.jsonl")
    parser.add_argument("--tokenizer_path", "--tokenizer", dest="tokenizer_path", type=str, default=TOKENIZER_PATH, help="Path to tokenizer.model")
    parser.add_argument("--output_dir", type=str, default=OUTPUT_DIR, help="Directory to save attention maps and metrics")
    parser.add_argument("--num_examples", type=int, default=3, help="Number of examples to visualize attention for")
    parser.add_argument("--font_path", type=str, default=None, help="Path to Devanagari font TTF")
    parser.add_argument("--device", type=str, default=None, help="Device to run analysis on ('cuda' or 'cpu')")
    args, _ = parser.parse_known_args()

    run_attention_analysis(args)

Loaded tokenizer from: /kaggle/input/notebooks/srinivasu2005/nepali-tokenizer/tokenizer/tokenizer.model
Loaded 3 examples from /kaggle/input/datasets/srinivasu2005/finetune-nepali/test.jsonl for attention analysis.
Attention analysis complete. Metrics and heatmaps saved to: eval_nepali_finetuned/attention
